# OT Syntactic Roles — Who Does What to Whom

Analysis of syntactic roles in the Hebrew OT using MACULA `subjref` links.
Given one or more Strong's numbers, the `role_search` module finds all verb
tokens whose grammatical subject resolves to those entities — enabling questions
like 'What verbs does YHWH take as subject?' and 'What does YHWH act upon?'

This notebook also covers loading and querying the OT Hebrew syntax data
from the MACULA Hebrew WLC corpus.

**Sections:**
1. Loading OT Hebrew Syntax Data (load_syntax_ot, query_syntax_ot)
2. What YHWH+Elohim Does (subject_verbs OT)
3. What YHWH Acts Upon (print_object_summary OT)
4. Who Does What to Whom (verb_subjects)
5. Syntactic Role Charts and Reports

In [ ]:
# @title Colab setup (runs only on Google Colab)
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess, os
    # Clone the repo so all source and data paths work
    if not os.path.isdir("/content/berean-bible-bots"):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/dnovick/berean-bible-bots.git",
             "/content/berean-bible-bots"],
            check=True,
        )
    os.chdir("/content/berean-bible-bots")
    sys.path.insert(0, "/content/berean-bible-bots/src")
    # Install Python dependencies
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         "binder/requirements.txt"],
        check=True,
    )
    # Download processed data files (~295 MB, one-time)
    subprocess.run(["bash", "binder/postBuild"], check=True)
    print("Colab environment ready.")


In [ ]:
import sys
sys.path.insert(0, '../../../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print('Ready.')

---
## 1. Loading OT Hebrew Syntax Data

The `syntax_ot` module wraps the MACULA Hebrew WLC data (475,911 tokens across
930 per-chapter lowfat XML files). Each word carries:

- **`type_`** — clause type: `wayyiqtol`, `qatal`, `yiqtol`, `imperative`, `participle`, `infinitive`, `nominal`, etc.
- **`stem`** — binyan: `qal`, `niphal`, `piel`, `pual`, `hiphil`, `hophal`, `hithpael`
- **`role`** — syntactic role (same codes as NT: s/v/o/...)
- **`subjref`** / **`participantref`** — xml_id links to grammatical subject/participant
- **`greek`** / **`greek_g`** — the LXX Greek word and Strong's number for each token
- **`lang`** — `H` (Hebrew) or `A` (Aramaic)

In [ ]:
from bible_grammar import load_syntax_ot, query_syntax_ot

df_ot = load_syntax_ot()
print(f'OT syntax tokens: {len(df_ot):,}')
print(f'Columns: {list(df_ot.columns)}')

In [ ]:
# Genesis 1:1 — "In the beginning God created the heavens and the earth"
gen1 = query_syntax_ot(book='Gen', chapter=1, verse=1)
gen1[['ref', 'text', 'lemma', 'strong_h', 'role', 'stem', 'type_', 'greek', 'greek_g']]

In [ ]:
# Clause type distribution across the OT — the backbone of Hebrew grammar
df_ot[df_ot['pos'] == 'verb']['type_'].value_counts().head(15)

In [ ]:
# wayyiqtol (narrative waw-consecutive past) in Genesis
wayyiqtol_gen = query_syntax_ot(book='Gen', tense='wayyiqtol')
print(f'wayyiqtol verbs in Genesis: {len(wayyiqtol_gen):,}')
wayyiqtol_gen[['ref', 'text', 'lemma', 'gloss', 'stem']].head(10)

In [ ]:
# Aramaic sections: Daniel 2:4-7:28, Ezra 4:8-6:18, 7:12-26
aramaic = query_syntax_ot(lang='A')
print(f'Aramaic tokens: {len(aramaic):,}')
aramaic.groupby('book').size().sort_values(ascending=False)

---
## 2. What YHWH+Elohim Does

The `role_search` module answers 'who does what to whom' by following MACULA
`subjref` links. This enables questions like:
- What verbs does YHWH/Elohim take as subject across the entire OT?
- What verbs are *only* ever attributed to divine subjects?
- How does divine action language in the OT compare to the NT?

In [ ]:
from bible_grammar import subject_verbs, print_role_summary

# What does God (YHWH + Elohim) do in the OT?
print_role_summary(['H3068', 'H0430'], corpus='OT', top_n=20, label='YHWH+Elohim')

In [ ]:
# As a DataFrame — includes lemma, gloss, stem, LXX Greek equivalent
df_god_ot = subject_verbs(['H3068', 'H0430'], corpus='OT', top_n=30)
df_god_ot[['lemma', 'gloss', 'stem', 'greek', 'greek_g', 'count']].head(20)

In [ ]:
# YHWH's verbs in Isaiah only
print_role_summary(['H3068'], corpus='OT', books=['Isa'],
                   top_n=15, label='YHWH (Isaiah)')

---
## 3. What YHWH Acts Upon

The symmetric complement to subject search: given an entity's Strong's number(s),
find what verbs are performed *on* that entity, and what objects verbs
with that subject act upon.

**OT method:** parses the MACULA Hebrew verb `frame` column (`A0`=agent, `A1`=patient)
to find verb-object triples when `A0` resolves to the target entity.

In [ ]:
from bible_grammar import print_object_summary

# What does YHWH+Elohim act upon in the OT?
print_object_summary(['H3068', 'H0430'], corpus='OT', top_n=20, label='YHWH+Elohim')

In [ ]:
# YHWH's objects in Isaiah — the Servant passages and divine speech
print_object_summary(['H3068'], corpus='OT', books=['Isa'], top_n=20, label='YHWH (Isaiah)')

---
## 4. Who Does What to Whom (verb_subjects)

The reverse of subject search: given a Strong's number for a *verb* root,
find what entities take that verb as their grammatical subject.
The classic biblical Hebrew question: does any human ever take בָּרָא (create)
as subject? The answer from the data is conclusive.

In [ ]:
from bible_grammar import verb_subjects

# Who does bara (H1254, to create) take as its grammatical subject?
# Classical question: does any human ever take bara as subject?
bara_subjects = verb_subjects('H1254', corpus='OT')
print('Grammatical subjects of bara (create) in the OT:')
bara_subjects

In [ ]:
from bible_grammar import object_verbs

# What verbs are performed ON Israel (H3478)?
# Answers: what does God do to Israel? what do enemies do to Israel?
ov = object_verbs('H3478', corpus='OT')
print('Verbs performed ON Israel:')
ov.head(15)

In [ ]:
from bible_grammar import subject_objects

# Raw DataFrame: all verb+object pairs for God in the OT
df_obj = subject_objects(['H3068', 'H0430'], corpus='OT', top_n=30)
df_obj[['verb_lemma', 'verb_gloss', 'obj_lemma', 'obj_gloss', 'obj_strong_h', 'count']].head(20)

---
## 5. Syntactic Role Charts and Reports

Visual and Markdown outputs for syntactic role analysis.
The chart compares top verbs for YHWH+Elohim across the OT.
The full report includes book distribution and cross-testament comparison.

In [ ]:
from bible_grammar import role_chart
from IPython.display import Image

chart_path = role_chart(
    ['H3068', 'H0430'], corpus='OT', top_n=20,
    label='YHWH+Elohim',
    output_path='../../../output/charts/ot/names/role-yhwh-elohim-ot.png'
)
print(f'Chart saved: {chart_path}')
Image(chart_path)

In [ ]:
from bible_grammar import divine_action_comparison
from IPython.display import Image

# Side-by-side: God's verbs in OT Hebrew vs NT Greek
ot_df, nt_df, chart_path = divine_action_comparison(
    output_path='../../../output/charts/both/names/divine-action-ot-nt.png'
)
print(f'Chart saved: {chart_path}')
Image(chart_path)

In [ ]:
from bible_grammar import role_report

# Generate full Markdown report: top verbs, book distribution, OT/NT comparison
report_path = role_report(
    ['H3068', 'H0430'], corpus='OT', top_n=30,
    label='YHWH+Elohim',
    output_dir='../../../output/reports/ot/lexicon',
    include_cross_testament=True
)
print(f'Report: {report_path}')